<a href="https://colab.research.google.com/github/Siva-p-11/Gpu-Computing-Colab/blob/main/Matrix_Addition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Fri Aug 28 08:03:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile matrix_add.cu

#include <stdio.h>
#include <cuda_runtime.h>

#define ROWS 4
#define COLS 4

// CUDA Kernel
__global__ void matrixAdd(float *A, float *B, float *C, int rows, int cols)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < rows && col < cols)
    {
        int index = row * cols + col;
        C[index] = A[index] + B[index];
    }
}

int main()
{
    int size = ROWS * COLS * sizeof(float);

    // Host matrices
    float h_A[ROWS][COLS];
    float h_B[ROWS][COLS];
    float h_C[ROWS][COLS];

    // Initialize matrices
    for (int i = 0; i < ROWS; i++)
    {
        for (int j = 0; j < COLS; j++)
        {
            h_A[i][j] = i + j;
            h_B[i][j] = (i + j) * 2;
        }
    }

    // Device matrices
    float *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    // Copy matrices from CPU to GPU
    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    // Define block and grid dimensions
    dim3 threadsPerBlock(16, 16);

    dim3 blocksPerGrid(
        (COLS + threadsPerBlock.x - 1) / threadsPerBlock.x,
        (ROWS + threadsPerBlock.y - 1) / threadsPerBlock.y
    );

    // Launch CUDA kernel
    matrixAdd<<<blocksPerGrid, threadsPerBlock>>>(
        d_A, d_B, d_C, ROWS, COLS
    );

    // Wait for GPU to finish
    cudaDeviceSynchronize();

    // Copy result from GPU to CPU
    cudaMemcpy(h_C, d_C, size, cudaMemcpyDeviceToHost);

    // Display Matrix A
    printf("Matrix A:\n");

    for (int i = 0; i < ROWS; i++)
    {
        for (int j = 0; j < COLS; j++)
        {
            printf("%.1f ", h_A[i][j]);
        }
        printf("\n");
    }

    // Display Matrix B
    printf("\nMatrix B:\n");

    for (int i = 0; i < ROWS; i++)
    {
        for (int j = 0; j < COLS; j++)
        {
            printf("%.1f ", h_B[i][j]);
        }
        printf("\n");
    }

    // Display Result
    printf("\nMatrix A + Matrix B:\n");

    for (int i = 0; i < ROWS; i++)
    {
        for (int j = 0; j < COLS; j++)
        {
            printf("%.1f ", h_C[i][j]);
        }
        printf("\n");
    }

    // Free GPU memory
    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing matrix_add.cu


In [3]:
!nvcc matrix_add.cu -o matrix_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./matrix_add

Matrix A:
0.0 1.0 2.0 3.0 
1.0 2.0 3.0 4.0 
2.0 3.0 4.0 5.0 
3.0 4.0 5.0 6.0 

Matrix B:
0.0 2.0 4.0 6.0 
2.0 4.0 6.0 8.0 
4.0 6.0 8.0 10.0 
6.0 8.0 10.0 12.0 

Matrix A + Matrix B:
0.0 3.0 6.0 9.0 
3.0 6.0 9.0 12.0 
6.0 9.0 12.0 15.0 
9.0 12.0 15.0 18.0 
